# 03b — GAN v2 d'inpainting

Même protocole que le notebook 03 (v1), avec trois changements (voir `configs/gan_v2.yaml`) :
- **blocs résiduels dilatés** (dilatations 2, 4, 8, 16) au niveau 16×16 du U-Net, pour voir plus loin dans l'image ;
- **300 epochs** au lieu de 150 ;
- meilleur modèle choisi sur le **SSIM** de validation au lieu du PSNR.

À lancer sur Kaggle avec un **GPU** et **Internet activé**. Entrées : `zellige_128.npz` et `masques_test_128.npz`.
Tous les fichiers produits portent le suffixe `_v2`.

## 1. Récupération du code et des données

In [ ]:
import os, sys, glob

REPO = "/kaggle/working/inpainting-patrimoine"
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/ayakorchi1/inpainting-patrimoine.git {REPO}
sys.path.insert(0, REPO)
os.chdir(REPO)

import numpy as np, torch
from src.data import trouver_npz

NPZ = trouver_npz("zellige_128.npz")
NPZ_MASQUES = trouver_npz("masques_test_128.npz")
SORTIE = "/kaggle/working"
CONFIG = "configs/gan_v2.yaml"
EXP = "gan_v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Données :", NPZ)
print("Masques :", NPZ_MASQUES)
print("Appareil :", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

## 2. Entraînement (hyperparamètres dans `configs/gan_v2.yaml`)

In [ ]:
from src.train_gan import entrainer

historique, infos = entrainer(CONFIG, NPZ, dossier=SORTIE)
infos

## 3. Courbes des pertes

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

h = pd.read_csv(f"{SORTIE}/historique_{EXP}.csv")
v = h.dropna(subset=["psnr_val"])
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
axes[0].plot(h.epoch, h.perte_D); axes[0].set_title("Perte du discriminateur (hinge)")
axes[1].plot(h.epoch, h.adverse); axes[1].set_title("Perte adverse du générateur")
axes[2].plot(h.epoch, h.l1); axes[2].set_title("L1 dans le trou")
axes[3].plot(v.epoch, v.psnr_val, marker="o"); axes[3].set_title("PSNR validation (dB)")
axes[4].plot(v.epoch, v.ssim_val, marker="o"); axes[4].set_title("SSIM validation (critère de sélection)")
axes[4].axvline(infos["meilleur_epoch"], color="red", ls="--", label=f"meilleur : epoch {infos['meilleur_epoch']}")
axes[4].legend()
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(f"{SORTIE}/gan_courbes_v2.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Évaluation sur le test (143 images × 9 masques fixes)
Même protocole que la baseline et la v1 : mêmes masques, mêmes fonctions `psnr_trou` et `ssim_img`, bruit fixe (graine 0).

In [ ]:
import time
from src.train_gan import charger_generateur, completer
from src.models.gan import compter_parametres
from src.metrics import psnr_trou, ssim_img

G, ckpt = charger_generateur(f"{SORTIE}/{EXP}_meilleur.pt", device)
print(f"Meilleur modèle : epoch {ckpt['epoch']} | SSIM val {ckpt['ssim_val']:.3f} | PSNR val {ckpt['psnr_val']:.2f} dB"
      f" | {compter_parametres(G)/1e6:.2f} M paramètres")

X_test = np.load(NPZ)["test"]
M = dict(np.load(NPZ_MASQUES))

lignes = []
for nom, masques in M.items():
    type_, pct = nom.rsplit("_", 1)
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    R = completer(G, X_test, masques, graine=0, device=device)
    if device == "cuda": torch.cuda.synchronize()
    temps_ms = (time.time() - t0) / len(X_test) * 1000
    ps = [psnr_trou(o, r, m) for o, r, m in zip(X_test, R, masques)]
    ss = [ssim_img(o, r) for o, r in zip(X_test, R)]
    lignes.append(dict(methode="gan_v2", type=type_, trou=int(pct),
                       PSNR=np.mean(ps), SSIM=np.mean(ss), temps_ms=temps_ms))

res = pd.DataFrame(lignes)
res.to_csv(f"{SORTIE}/resultats_gan_v2.csv", index=False)
print("PSNR moyen (dB) :")
display(res.pivot_table(index="type", columns="trou", values="PSNR").round(2))
print("SSIM moyen :")
display(res.pivot_table(index="type", columns="trou", values="SSIM").round(3))
print(f"Temps moyen : {res.temps_ms.mean():.2f} ms / image")

## 5. Diversité : 4 propositions par image
4 bruits différents par image ; écart moyen entre les propositions dans le trou (échelle 0-255). **0 = identiques.**

In [ ]:
from src.metrics import diversite_trou

GRAINES = [0, 1000, 2000, 3000]
lignes_div = []
for nom, masques in M.items():
    type_, pct = nom.rsplit("_", 1)
    props = [completer(G, X_test, masques, graine=g, device=device) for g in GRAINES]
    d = [diversite_trou([p[i] for p in props], masques[i]) for i in range(len(X_test))]
    lignes_div.append(dict(methode="gan_v2", type=type_, trou=int(pct), diversite=np.mean(d)))

div = pd.DataFrame(lignes_div)
div.to_csv(f"{SORTIE}/resultats_gan_diversite_v2.csv", index=False)
display(div.pivot_table(index="type", columns="trou", values="diversite").round(2))
if div.diversite.mean() < 1:
    print("⚠ Diversité quasi nulle : le générateur ignore le bruit (effondrement de modes).")

## 6. Comparaison Telea / GAN v1 / GAN v2
Telea : `results/resultats_baseline.csv` ; v1 : `results/resultats_gan_v1.csv` et `results/resultats_gan_diversite_v1.csv`
(fichiers du dépôt) ; v2 : ce notebook. Telea est déterministe : sa diversité vaut 0 par construction.

In [ ]:
telea = pd.read_csv("results/resultats_baseline.csv").query("methode == 'telea'")
v1 = pd.read_csv("results/resultats_gan_v1.csv").assign(methode="gan_v1")
div_v1 = pd.read_csv("results/resultats_gan_diversite_v1.csv").assign(methode="gan_v1")

perf = pd.concat([telea, v1, res])[["methode", "type", "trou", "PSNR", "SSIM", "temps_ms"]]
diversites = pd.concat([telea[["methode", "type", "trou"]].assign(diversite=0.0), div_v1, div])
comp = perf.merge(diversites, on=["methode", "type", "trou"])
ordre = {"telea": 0, "gan_v1": 1, "gan_v2": 2}
comp = comp.sort_values(["type", "trou", "methode"], key=lambda s: s.map(ordre) if s.name == "methode" else s)
comp.to_csv(f"{SORTIE}/comparaison_telea_gan_v1_v2.csv", index=False)

tableau = comp.pivot_table(index=["type", "trou"], columns="methode",
                           values=["PSNR", "SSIM", "diversite"])[["PSNR", "SSIM", "diversite"]]
tableau = tableau.reindex(columns=["telea", "gan_v1", "gan_v2"], level=1)
display(tableau.round({("PSNR", m): 2 for m in ordre} | {("SSIM", m): 3 for m in ordre}
                      | {("diversite", m): 2 for m in ordre}))

print("Moyennes sur les 9 configurations :")
moy = comp.groupby("methode")[["PSNR", "SSIM", "diversite", "temps_ms"]].mean()
display(moy.loc[["telea", "gan_v1", "gan_v2"]].round(3))

gains = res.set_index(["type", "trou"])[["PSNR", "SSIM"]] - v1.set_index(["type", "trou"])[["PSNR", "SSIM"]]
print(f"v2 - v1 : PSNR {gains.PSNR.mean():+.2f} dB (meilleure sur {(gains.PSNR > 0).sum()}/9) | "
      f"SSIM {gains.SSIM.mean():+.3f} (meilleure sur {(gains.SSIM > 0).sum()}/9)")

## 7. Figures
Le modèle v1 n'est pas chargé ici : la grille v1 (`results/gan_exemples_v1.png`, même image n° 5, mêmes masques)
est affichée à côté de la nouvelle grille v2 pour comparer.

In [ ]:
import cv2
import matplotlib.image as mpimg

i = 5   # même image que pour la baseline et la v1
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for row, pct in enumerate([10, 20, 40]):
    m = M[f"central_{pct}"][i]
    abimee = X_test[i].copy(); abimee[m == 1] = 255
    telea_img = cv2.inpaint(X_test[i], m, 3, cv2.INPAINT_TELEA)
    gan = completer(G, X_test[i:i+1], m[None], graine=0, device=device)[0]
    for col, (im, titre) in enumerate([(abimee, f"Abîmée {pct}%"), (telea_img, "Telea"),
                                       (gan, "GAN v2"), (X_test[i], "Originale")]):
        axes[row, col].imshow(im); axes[row, col].set_title(titre); axes[row, col].axis("off")
plt.tight_layout()
plt.savefig(f"{SORTIE}/gan_exemples_v2.png", dpi=150, bbox_inches="tight"); plt.show()

# v1 (image enregistrée) à côté de v2
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, chemin, titre in [(axes[0], "results/gan_exemples_v1.png", "GAN v1"),
                          (axes[1], f"{SORTIE}/gan_exemples_v2.png", "GAN v2")]:
    ax.imshow(mpimg.imread(chemin)); ax.set_title(titre, fontsize=16); ax.axis("off")
plt.tight_layout()
plt.savefig(f"{SORTIE}/gan_comparaison_v1_v2.png", dpi=120, bbox_inches="tight"); plt.show()

# 4 propositions pour une même image (trou central 40 %)
m = M["central_40"][i]
abimee = X_test[i].copy(); abimee[m == 1] = 255
props = [completer(G, X_test[i:i+1], m[None], graine=g, device=device)[0] for g in GRAINES]
fig, axes = plt.subplots(1, 6, figsize=(18, 3.4))
for ax, (im, titre) in zip(axes, [(abimee, "Abîmée 40%")] +
                           [(p, f"Proposition {k+1}") for k, p in enumerate(props)] +
                           [(X_test[i], "Originale")]):
    ax.imshow(im); ax.set_title(titre); ax.axis("off")
plt.tight_layout()
plt.savefig(f"{SORTIE}/gan_propositions_v2.png", dpi=150, bbox_inches="tight"); plt.show()

print("À télécharger depuis /kaggle/working puis à mettre dans results/ du dépôt :")
print("resultats_gan_v2.csv, resultats_gan_diversite_v2.csv, historique_gan_v2.csv, memoire_gpu_gan_v2.json,")
print("comparaison_telea_gan_v1_v2.csv, gan_courbes_v2.png, gan_exemples_v2.png, gan_comparaison_v1_v2.png,")
print("gan_propositions_v2.png (PAS les fichiers .pt). Vérifier que l'extension n'est pas devenue .txt.")